In [ ]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Exploratory Data Analysis (EDA)
This notebook performs comprehensive exploratory data analysis on the dataset to understand data patterns, distributions, relationships, and quality.

## 2. Load and Inspect the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../data/churn.csv')  # Update path as needed

# Display basic information
print("Dataset Shape:", df.shape)
print("\nColumn Names and Types:")
print(df.dtypes)
print("\nFirst Few Rows:")
print(df.head())
print("\nDataset Info:")
df.info()

## 3. Data Quality Assessment

In [ ]:
# Check for missing values
print("Missing Values Count:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values")

# Check for duplicates
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

# Data quality report
print("\n=== DATA QUALITY REPORT ===")
print(f"Total Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum()}")
print(f"Data Completeness: {(1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100:.2f}%")

## 4. Univariate Analysis

In [ ]:
# Univariate Analysis - Numerical Columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(len(numerical_cols), 2, figsize=(14, 5*len(numerical_cols)))
if len(numerical_cols) == 1:
    axes = axes.reshape(1, -1)

for idx, col in enumerate(numerical_cols):
    # Histogram
    axes[idx, 0].hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    axes[idx, 0].set_title(f'Distribution of {col}')
    axes[idx, 0].set_xlabel(col)
    axes[idx, 0].set_ylabel('Frequency')
    
    # Box plot
    axes[idx, 1].boxplot(df[col])
    axes[idx, 1].set_title(f'Box Plot of {col}')
    axes[idx, 1].set_ylabel(col)

plt.tight_layout()
plt.show()

# Univariate Analysis - Categorical Columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols[:6]:  # Display first 6 categorical columns
    fig, ax = plt.subplots(figsize=(10, 4))
    df[col].value_counts().plot(kind='bar', ax=ax, edgecolor='black')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 5. Bivariate Analysis

In [ ]:
# Bivariate Analysis - Numerical vs Numerical
if len(numerical_cols) >= 2:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(df[numerical_cols[0]], df[numerical_cols[1]], alpha=0.5)
    ax.set_xlabel(numerical_cols[0])
    ax.set_ylabel(numerical_cols[1])
    ax.set_title(f'Relationship between {numerical_cols[0]} and {numerical_cols[1]}')
    plt.show()

# Bivariate Analysis - Categorical vs Numerical
if categorical_cols and numerical_cols:
    fig, ax = plt.subplots(figsize=(10, 6))
    df.boxplot(column=numerical_cols[0], by=categorical_cols[0], ax=ax)
    ax.set_title(f'{numerical_cols[0]} by {categorical_cols[0]}')
    plt.suptitle('')  # Suppress the automatic title
    plt.show()

## 6. Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = df[numerical_cols].corr()

# Visualize correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation Matrix - Numerical Variables')
plt.tight_layout()
plt.show()

# Print high correlations
print("\nHigh Correlations (> 0.7 or < -0.7):")
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.7:
            print(f"{correlation_matrix.columns[i]} - {correlation_matrix.columns[j]}: {correlation_matrix.iloc[i, j]:.3f}")

## 7. Data Distribution Visualization

In [ ]:
# Distribution Visualization using KDE and Violin Plots
for col in numerical_cols[:4]:  # Display first 4 numerical columns
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # KDE Plot
    df[col].plot(kind='kde', ax=axes[0], color='blue', linewidth=2)
    axes[0].set_title(f'KDE Plot of {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Density')
    
    # Violin Plot (if categorical column exists)
    if categorical_cols:
        sns.violinplot(data=df, y=col, x=categorical_cols[0], ax=axes[1])
        axes[1].set_title(f'Violin Plot: {col} by {categorical_cols[0]}')
    else:
        axes[1].remove()
    
    plt.tight_layout()
    plt.show()

## 8. Summary Statistics

In [ ]:
# Descriptive Statistics for Numerical Columns
print("=== DESCRIPTIVE STATISTICS ===\n")
print(df[numerical_cols].describe().T)

print("\n=== SKEWNESS AND KURTOSIS ===\n")
for col in numerical_cols:
    skewness = stats.skew(df[col].dropna())
    kurtosis = stats.kurtosis(df[col].dropna())
    print(f"{col}:")
    print(f"  Skewness: {skewness:.4f}")
    print(f"  Kurtosis: {kurtosis:.4f}\n")

# Categorical Features Summary
print("\n=== CATEGORICAL FEATURES SUMMARY ===\n")
for col in categorical_cols:
    print(f"{col}:")
    print(df[col].value_counts())
    print(f"  Unique Values: {df[col].nunique()}\n")